In [ ]:
## LODO Test Script
# This script tests the functionality of the LODO (Leave-One-Dataset-Out) module.

In [ ]:
# Step 1: Load necessary libraries
import pandas as pd
import numpy as np
import tensorflow as tf
import os
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score

e:\conda-envs\tensorflow\lib\site-packages\scipy\__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.24.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [ ]:
# Set working directory
os.chdir(r"D:\结直肠癌肝转移Biomarker 诊断\投稿\Brief in Bioinformatics\Github\Samples merge and split")

# 1. Load data
X_all = pd.read_csv("expression_data_combat_corrected.csv", index_col=0)
# 根据截图，samples_for_LODO_test.csv 的 ID 在 'sample_id' 列
meta = pd.read_csv("samples_for_LODO_test.csv", index_col='sample_id')

# 2. Sample alignment: Intersection of expression matrix and metadata
common_samples = X_all.index.intersection(meta.index)
X_all = X_all.loc[common_samples]
meta = meta.loc[common_samples]

# 3. Label mapping: Mapping 'primary' to 0 and 'metastasis' to 1
y = meta["group"].map({"primary": 0, "metastasis": 1}).values

# 4. Dataset labels for LODO (Leave-One-Dataset-Out)
dataset_labels = meta["batch"].values

# 5. Feature selection: Filter genes not present in the expression matrix
with open("functional_genes_620.txt") as f:
    genes_list = [g.strip() for g in f.readlines()]

# Select only available genes to prevent KeyError
available_genes = [g for g in genes_list if g in X_all.columns]
missing_genes = set(genes_list) - set(available_genes)

if missing_genes:
    print(f"Note: {len(missing_genes)} genes were not found in the expression matrix and have been skipped.")
    print("Example of missing genes:", list(missing_genes)[:5])

X_all = X_all[available_genes]

print(f"Final samples for analysis: {X_all.shape[0]}, Number of genes: {X_all.shape[1]}")

In [ ]:
# Step 2: Prepare LODO parameters
unique_datasets = meta["batch"].unique()
lodo_results = []

In [5]:
# -------------------------------------------------
# Define CNN architecture (same as manuscript)
# -------------------------------------------------
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input, Conv1D, MaxPooling1D, Flatten

def create_cnn_model(input_shape):
    model = Sequential([
        Input(shape=input_shape),
        Conv1D(filters=32, kernel_size=5, activation='relu'),
        MaxPooling1D(pool_size=2),
        Dropout(0.3),
        Conv1D(filters=64, kernel_size=5, activation='relu'),
        MaxPooling1D(pool_size=2),
        Dropout(0.4),
        Flatten(),
        Dense(64, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

In [ ]:
# Step 3: Main LODO Loop
for left_out in unique_datasets:
    print(f"\n===== LODO: Leaving out {left_out} =====")

    try:
        # -------------------------
        # Split by dataset
        # -------------------------
        test_idx = meta["batch"] == left_out
        train_idx = ~test_idx

        X_train_all = X_all.loc[train_idx]
        y_train_all = y[train_idx]

        X_test = X_all.loc[test_idx]
        y_test = y[test_idx]

        print(f"Train samples: {X_train_all.shape[0]}")
        print(f"Test samples ({left_out}): {X_test.shape[0]}")

        # -------------------------
        # Inner 5-fold CV
        # -------------------------
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        fold_predictions = []

        for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train_all, y_train_all), 1):
            print(f"  Fold {fold}")

            X_tr = X_train_all.iloc[tr_idx]
            y_tr = y_train_all[tr_idx]
            X_val = X_train_all.iloc[val_idx]
            y_val = y_train_all[val_idx]

            scaler = StandardScaler()
            X_tr_scaled = scaler.fit_transform(X_tr)
            X_val_scaled = scaler.transform(X_val)

            X_tr_cnn = np.expand_dims(X_tr_scaled, axis=-1)
            X_val_cnn = np.expand_dims(X_val_scaled, axis=-1)

            model = create_cnn_model((X_tr_cnn.shape[1], 1))
            model.fit(
                X_tr_cnn, y_tr,
                epochs=50,
                batch_size=32,
                validation_data=(X_val_cnn, y_val),
                verbose=0
            )

            # Predict on left-out dataset
            X_test_scaled = scaler.transform(X_test)
            X_test_cnn = np.expand_dims(X_test_scaled, axis=-1)
            preds = model.predict(X_test_cnn, verbose=0).flatten()
            fold_predictions.append(preds)

        # -------------------------
        # Ensemble + Evaluation
        # -------------------------
        y_test_proba = np.mean(fold_predictions, axis=0)

        auc = roc_auc_score(y_test, y_test_proba) if len(np.unique(y_test)) > 1 else np.nan
        acc = accuracy_score(y_test, (y_test_proba > 0.5).astype(int))
        prec = precision_score(y_test, (y_test_proba > 0.5).astype(int), zero_division=0)
        rec = recall_score(y_test, (y_test_proba > 0.5).astype(int), zero_division=0)

        # Store results for each iteration
        lodo_results.append({
            "Left_out_dataset": left_out,
            "N_test": len(y_test),
            "AUC": auc,
            "Accuracy": acc,
            "Precision": prec,
            "Recall": rec
        })

        print(f"LODO result for {left_out}: AUC={auc:.3f}, Recall={rec:.3f}")

    except Exception as e:
        # Error handling to prevent single dataset failure from stopping the loop
        print(f"❌ LODO failed for {left_out}: {e}")

        lodo_results.append({
            "Left_out_dataset": left_out,
            "N_test": len(y_test),
            "AUC": np.nan,
            "Accuracy": np.nan,
            "Precision": np.nan,
            "Recall": np.nan
        })

In [ ]:
# Step 4: Summarization and Output
lodo_df = pd.DataFrame(lodo_results)

print("\n===== LODO Summary =====")
print(lodo_df)

print(
    f"\nMean ± SD AUC: "
    f"{lodo_df['AUC'].mean():.3f} ± {lodo_df['AUC'].std():.3f}"
)

lodo_df = pd.DataFrame(lodo_results)

# Clean results by removing placeholder NaN rows if necessary
lodo_df = lodo_df.dropna(subset=["AUC"]).reset_index(drop=True)

print(
    f"Mean ± SD AUC: "
    f"{lodo_df['AUC'].mean():.3f} ± {lodo_df['AUC'].std():.3f}"
)

# Define save directory for results
save_dir = r"D:\结直肠癌肝转移Biomarker 诊断\投稿\Computational and Structural Biotechnology\Ready for Submit\For Submit\返修"
os.makedirs(save_dir, exist_ok=True)
# Export results to CSV
lodo_df.to_csv(
    os.path.join(save_dir, "LODO_Results_Summary.csv"),
    index=False
)
# Export metrics report to TXT
with open(os.path.join(save_dir, "LODO_Metrics_Report.txt"), "w") as f:
    f.write("===== LODO Summary =====\n")
    f.write(lodo_df.to_string())
    f.write(
        f"\n\nMean ± SD AUC: "
        f"{lodo_df['AUC'].mean():.3f} ± {lodo_df['AUC'].std():.3f}\n"
    )